In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from pyspark.sql.functions import col, year, month, when, sum, count, desc

In [5]:
# 1. Khởi tạo SparkSession
spark =  (SparkSession.builder 
    .appName("Retail_Bigdata_Project") 
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") 
    .getOrCreate())

# 2. Định nghĩa Schema (Ép kiểu dữ liệu chuẩn)
schema = StructType([
    StructField("Transaction ID", IntegerType(), True),
    StructField("Date", DateType(), True),
    StructField("Customer ID", StringType(), True),
    StructField("Gender", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Product Category", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Price per Unit", DoubleType(), True),
    StructField("Total Amount", DoubleType(), True)
])

# 3. Đọc file CSV với Schema định nghĩa
df_retail = spark.read.csv("D:\\Downloads\\Retail_Sales_Project\\Retail_Sales_Project.ipynb", header = True, schema = schema)

In [6]:
# 4. Data Transformation (Tạo các cột phân tích mới)
df_transformed =  df_retail \
    .withColumn("Year", year(col("Date"))) \
    .withColumn("Month", month(col("Date"))) \
    .withColumn("Age_Group",
                when(col("Age") < 25,"18-24")
                .when((col("Age") >= 25) & (col("Age") < 35), "25-35")
                .when((col("Age") > 35) & (col("Age") < 50), "36-50")
                .otherwise("50+")
                )  \
    .withColumn("Spend_Level",
                when(col("Total Amount") > 1000, "High")
                .when(col("Total Amount") >=500, "Medium")
                .otherwise("Low")
                )             

df_transformed.show(5)     
                                   
    
# 5. Lưu dữ liệu dưới dạng Parquet (Giả lập lưu trữ Big Data)
# Partition theo Year và Month để tăng tốc độ truy vấn sau này
df_transformed.write.mode("overwrite").partitionBy("Year", "Month").parquet("processed_data/retail_parquet")
print("Đã lưu Parquet thành công!")

+--------------+----+-----------+------+----+----------------+--------+--------------+------------+----+-----+---------+-----------+
|Transaction ID|Date|Customer ID|Gender| Age|Product Category|Quantity|Price per Unit|Total Amount|Year|Month|Age_Group|Spend_Level|
+--------------+----+-----------+------+----+----------------+--------+--------------+------------+----+-----+---------+-----------+
|          NULL|NULL|       NULL|  NULL|NULL|            NULL|    NULL|          NULL|        NULL|NULL| NULL|      50+|        Low|
|          NULL|NULL|       NULL|  NULL|NULL|            NULL|    NULL|          NULL|        NULL|NULL| NULL|      50+|        Low|
|          NULL|NULL|       NULL|  NULL|NULL|            NULL|    NULL|          NULL|        NULL|NULL| NULL|      50+|        Low|
|          NULL|NULL|       NULL|  NULL|NULL|            NULL|    NULL|          NULL|        NULL|NULL| NULL|      50+|        Low|
|          NULL|NULL|       NULL|  NULL|NULL|            NULL|    NUL

In [ ]:
# Đăng ký DataFrame thành một bảng tạm thời để viết SQL
df_transformed.createOrReplaceTempView("retial_sales")

# --- Câu hỏi 1: Doanh thu và Số lượng bán ra theo từng Danh mục sản phẩm? (Dùng Spark SQL) ---
print("1. Doanh thu theo Category")
revenue_by_category = spark.sql("""
    SELECT  
        `Product Category`
        , SUM(`Total Amount`) AS Total_Revenue
        , SUM(`Quantity`) AS Total_Quantity
    FROM    retial_sales
    GROUP BY `Product Category`
    ORDER BY Total_Revenue DESC
""")
revenue_by_category.show();

# --- Câu hỏi 2: Nhóm tuổi nào mua sắm nhiều nhất ở từng danh mục? (Dùng DataFrame API) ---
age_category_insights = spark.sql("""
    SELECT
        `Product Category`
        ,`Age_Group`
        , SUM(`Total Amount`) AS Total_Revenue
    FROM    retial_sales  
    GROUP BY    `Product Category`,`Age_Group`
    ORDER BY    `Product Category`, `Age_Group` 
""")
age_category_insights.show()

# --- Câu hỏi 3: Tìm ra Top 5 khách hàng VIP (Chi tiêu nhiều nhất)? ---
total_revenue_top_5 = spark.sql("""
    SELECT 
        `Customer ID`
        , COUNT(*) AS Number_Of_Order
        , SUM(`Total Amount`) AS Total_Revenue
    FROM    retial_sales
    GROUP BY    `Customer ID` 
    ORDER BY    Total_Revenue DESC
    LIMIT 5
""")
total_revenue_top_5.show()

1. Doanh thu theo Category
+----------------+-------------+--------------+
|Product Category|Total_Revenue|Total_Quantity|
+----------------+-------------+--------------+
|            NULL|         NULL|          NULL|
|           count|         NULL|          NULL|
|   DoubleType\n"|         NULL|          NULL|
+----------------+-------------+--------------+

+----------------+---------+-------------+
|Product Category|Age_Group|Total_Revenue|
+----------------+---------+-------------+
|            NULL|      50+|         NULL|
|   DoubleType\n"|      50+|         NULL|
|           count|      50+|         NULL|
+----------------+---------+-------------+

+------------------+---------------+-------------+
|       Customer ID|Number_Of_Order|Total_Revenue|
+------------------+---------------+-------------+
|       IntegerType|              1|         NULL|
|              NULL|            199|         NULL|
|             month|              1|         NULL|
|             True)|        

In [9]:
import streamlit as st
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from pyspark.sql.functions import col, year, month, when
import pandas as pd
import plotly.express as px

# 1. Cấu hình trang Streamlit (Phải đặt ở đầu tiên)
st.set_page_config(page_title="Retail Sales Dashboard", page_icon="🛒", layout="wide")

# 2. Tối ưu hóa: Dùng decorator của Streamlit để cache dữ liệu
# Tránh việc Spark phải đọc lại file mỗi khi người dùng click vào Dashboard
@st.cache_resource
def get_spark_session():
    return SparkSession.builder.appName("Retail_Streamlit").getOrCreate()

@st.cache_data
def load_and_transform_data():
    spark = get_spark_session()
    
    # Định nghĩa Schema
    schema = StructType([
        StructField("Transaction ID", IntegerType(), True),
        StructField("Date", DateType(), True),
        StructField("Customer ID", StringType(), True),
        StructField("Gender", StringType(), True),
        StructField("Age", IntegerType(), True),
        StructField("Product Category", StringType(), True),
        StructField("Quantity", IntegerType(), True),
        StructField("Price per Unit", DoubleType(), True),
        StructField("Total Amount", DoubleType(), True)
    ])
    
    # Đọc file CSV
    df = spark.read.csv("D:\\Downloads\\Retail_Sales_Project\\Retail_Sales_Project.ipynb", header=True, schema=schema)
    
    # ETL (Transform)
    df_transformed = df \
        .withColumn("Year", year(col("Date"))) \
        .withColumn("Month", month(col("Date"))) \
        .withColumn("Age_Group", 
            when(col("Age") < 25, "18-24")
            .when((col("Age") >= 25) & (col("Age") <= 35), "25-35")
            .when((col("Age") > 35) & (col("Age") <= 50), "36-50")
            .otherwise("50+"))
            
    # Trong Big Data thực tế, ta sẽ dùng Spark groupby rồi mới toPandas() 
    # Nhưng vì data nhỏ (1000 dòng), ta toPandas toàn bộ để Streamlit dễ filter tương tác
    return df_transformed.toPandas()

# ==========================================
# XÂY DỰNG GIAO DIỆN STREAMLIT
# ==========================================

# Gọi hàm load data
df_pd = load_and_transform_data()

# --- SIDEBAR (BỘ LỌC) ---
st.sidebar.image("https://cdn-icons-png.flaticon.com/512/3081/3081840.png", width=100)
st.sidebar.title("Bộ lọc dữ liệu")

# Lọc theo Giới tính
selected_gender = st.sidebar.multiselect(
    "Chọn Giới tính:", 
    options=df_pd["Gender"].unique(),
    default=df_pd["Gender"].unique()
)

# Lọc theo Nhóm tuổi
selected_age = st.sidebar.multiselect(
    "Chọn Nhóm tuổi:",
    options=df_pd["Age_Group"].unique(),
    default=df_pd["Age_Group"].unique()
)

# Áp dụng bộ lọc vào DataFrame
df_filtered = df_pd.query("Gender == @selected_gender & Age_Group == @selected_age")

# --- MAIN DASHBOARD ---
st.title("🛒 Retail Sales Dashboard")
st.markdown("*Dự án ứng dụng PySpark cho xử lý ETL và Streamlit để trực quan hóa dữ liệu.*")
st.divider()

# --- KPIs (Các chỉ số quan trọng) ---
col1, col2, col3, col4 = st.columns(4)
total_revenue = df_filtered["Total Amount"].sum()
total_transactions = df_filtered["Transaction ID"].nunique()
total_customers = df_filtered["Customer ID"].nunique()
avg_order_value = df_filtered["Total Amount"].mean()

col1.metric("💰 Tổng Doanh Thu", f"${total_revenue:,.0f}")
col2.metric("📦 Số Giao Dịch", f"{total_transactions}")
col3.metric("👥 Số Khách Hàng", f"{total_customers}")
col4.metric("🛒 Giá Trị Đơn Trung Bình", f"${avg_order_value:,.2f}")

st.divider()

# --- BIỂU ĐỒ TRỰC QUAN (PLOTLY INTERACTIVE) ---
chart_col1, chart_col2 = st.columns(2)

with chart_col1:
    st.subheader("Doanh thu theo Danh mục sản phẩm")
    # Group data (Pandas)
    category_rev = df_filtered.groupby("Product Category")["Total Amount"].sum().reset_index()
    fig1 = px.bar(category_rev, x="Product Category", y="Total Amount", 
                  color="Product Category", text_auto=True,
                  title="Doanh thu từng danh mục")
    st.plotly_chart(fig1, use_container_width=True)

with chart_col2:
    st.subheader("Tỷ lệ Giới tính Khách hàng")
    gender_count = df_filtered["Gender"].value_counts().reset_index()
    gender_count.columns = ["Gender", "Count"]
    fig2 = px.pie(gender_count, values="Count", names="Gender", 
                  hole=0.4, title="Phân bổ giới tính")
    st.plotly_chart(fig2, use_container_width=True)

st.divider()

st.subheader("📈 Xu hướng Doanh thu hàng tháng")
# Sort data theo tháng
monthly_trend = df_filtered.groupby("Month")["Total Amount"].sum().reset_index().sort_values("Month")
fig3 = px.line(monthly_trend, x="Month", y="Total Amount", markers=True, 
               title="Biến động doanh thu theo tháng (2023)",
               labels={"Month": "Tháng", "Total Amount": "Doanh Thu ($)"})
fig3.update_xaxes(tickmode='linear') # Hiển thị đủ 12 tháng
st.plotly_chart(fig3, use_container_width=True)

# --- HIỂN THỊ RAW DATA NẾU CẦN ---
with st.expander("🔍 Bấm vào đây để xem dữ liệu thô (Đã qua xử lý PySpark)"):
    st.dataframe(df_filtered)

2026-09-15 09:52:08.069 No runtime found, using MemoryCacheStorageManager
2026-09-15 09:52:08.072 No runtime found, using MemoryCacheStorageManager
